# Atlan API SDK

In [1]:
import TELTools as TT
import requests
import json
import pandas as pd
from datetime import datetime as dt
import re
import ast
import time
import os

from pyatlan.client.atlan import AtlanClient
from pyatlan.model.enums import LineageDirection, EntityStatus, AtlanTypeCategory, AnnouncementType, AtlanConnectorType
from pyatlan.model.assets import AtlasGlossary, Asset, Schema, Table, Column, View, Connection, DbtModel, Dbt, DbtProcess, CogniteAsset, DbtTest
from pyatlan.model.lineage import FluentLineage
from pyatlan.model.fluent_search import FluentSearch, CompoundQuery
from pyatlan.model.fields.atlan_fields import CustomMetadataField
from typing import TYPE_CHECKING, Any, Optional, Union
from pyatlan.model.core import Announcement
from pyatlan.model.custom_metadata import CustomMetadataDict
from pyatlan.cache.role_cache import RoleCache

In [2]:
atlan_token = TT.getAzureSecret('ATLAN-ADMIN-TOKEN').value
tenant_url = 'https://tel.atlan.com'

client = AtlanClient(
    base_url=tenant_url,
    api_key=atlan_token,
)
AtlanClient.set_default_client(client)

## User data

In [ ]:
users = client.user.get_all()

df = pd.DataFrame(columns=['Username', 'Email', 'Enabled', 'Verified', 'Role', 'Created', 'Last Login'])

for user in users:
    username = user.username
    email = user.email
    enabled = user.enabled
    verified = user.email_verified
    roles = user.roles or []
    role = '; '.join([role.replace('$', '') for role in roles if role.startswith('$')])
    default_roles = user.default_roles or []
    default_role = '; '.join([role.replace('$', '') for role in default_roles if role.startswith('$')])
    role = role or default_role
    created_timestamp = user.created_timestamp or 0
    created = dt.fromtimestamp(created_timestamp/1000)
    last_login_time = user.last_login_time or 0
    last_login = dt.fromtimestamp(last_login_time/1000)

    data = [
        username,
        email,
        enabled,
        verified,
        role,
        created,
        last_login
    ]
    df.loc[len(df)] = data
df

## Search requests

In [ ]:
# example search request parameters
request = (
    FluentSearch()
    # .where(CompoundQuery.active_assets())
    .where(CompoundQuery.asset_type(Connection))
    .where(Table.QUALIFIED_NAME.startswith('default/dbt/')) 
    # .where(Column.GUID.eq('e7d4ce83-8808-4a62-a224-7e8d2459fd35'))
    # .where(Asset.DBT_QUALIFIED_NAME.eq('x'))
    # .where(CustomMetadataField(set_name="Data Quality", attribute_name="DQ_CHECKS_PASSED").eq(False))
    # .where_not(Asset.ANNOUNCEMENT_TYPE.eq(AnnouncementType.ISSUE.value))
).to_request()
results = list(client.asset.search(request))
results

## DBT connections

In [ ]:
# dbt_connection = client.get_asset_by_guid(guid='7b659689-7130-48cb-95be-27b4197b9c32', asset_type=Connection)
dbt_connection = client.get_asset_by_qualified_name(qualified_name='default/dbt/1723739048', asset_type=Connection)
# dbt_connection = client.find_connections_by_name(name='telus', connector_type=AtlanConnectorType.DBT)[0]
dbt_connection

In [ ]:
# Tests creating different types of Atlan dbt assets (ie model, tests, processes, etc)

# table_a = client.get_asset_by_qualified_name('default/snowflake/1650402931/COMMON/KANBANIZE/SPRINT_USER_METRICS', asset_type=Table)
# table_b = client.get_asset_by_qualified_name('default/snowflake/1650402931/COMMON/KANBANIZE/SPRINT_METRICS', asset_type=Table)
# table_c = client.get_asset_by_qualified_name('default/snowflake/1650402931/PUB/COMMON_KANBANIZE/SPRINT_USER_METRICS', asset_type=View)
# print(table_a)
# print(table_b)
# print(table_c)

# request = FluentLineage(
#     starting_guid='d98d152f-12f3-4569-8749-dd8f89a1ac9e',
#     direction=LineageDirection.DOWNSTREAM,
#     includes_in_results=[
#         Asset.TYPE_NAME.in_lineage.eq('Process'),
#     ],
# ).request
# lineage_response = list(client.asset.get_lineage_list(request))
# process_ab = lineage_response[1]


# dbt_connection_qualified_name = 'default/dbt/1723739048'
# newAsset = Dbt.updater(
#      name='DbtTest',
#      qualified_name=dbt_connection_qualified_name + '/DbtTest'
#  )
# newAsset.connection_qualified_name = dbt_connection_qualified_name #supply connection qualified name
# newAsset.connector_name = AtlanConnectorType.DBT #replace with connector name you are using
# newAsset.connection_name = 'telus' #replace with connection name you are using
# response = client.asset.save(newAsset)

# newDBT = client.get_asset_by_qualified_name('default/dbt/1723739048/DbtTest', asset_type=Dbt)
# newDBT.dbt_alias = 'ex_dbt_alias'
# newDBT.dbt_project_name = 'dbt_project'
# newDBT.dbt_tags = {'tag2', 'tag5'}
# newDBT.dbt_project_name = 'dbt_project'


# response = client.asset.save(newDBT)

# newModel = client.get_asset_by_guid('61a12f9d-b71e-4735-90ae-6cf7f28faa3d', asset_type=DbtModel)
# newModel.name = 'TestAsset2'
# newModel.description = 'part 7'
# newModel.dbt_raw_s_q_l = 'insert into table select 1'
# newModel.dbt_compiled_s_q_l = 'select 1'
# newModel.dbt_status = 'SUCCESS'
# newModel.dbt_materialization_type = 'merge'
# newModel.dbt_model_compile_started_at = dt.now()
# newModel.dbt_model_compile_completed_at = dt.now()
# newModel.dbt_model_execute_started_at = dt.now()
# newModel.dbt_model_execute_completed_at = dt.now()
# newModel.dbt_model_execution_time = 1.1
# newModel.dbt_model_run_elapsed_time = 1.2
# newModel.dbt_model_sql_assets = [table_a]
# newModel.sql_asset = table_b

# response = client.asset.save(newModel)

# newProcess = client.get_asset_by_qualified_name('default/dbt/1723739048/TestProcess', asset_type=DbtProcess)
# newProcess.dbt_project_name = 'dbt_project'
# newProcess.dbt_process_job_status = 'ERROR' # ERROR or SUCCESS are automatically understood by Atlan which is able to color coordinate
# newProcess.dbt_tags = {'tag1', 'tag2'}
# newProcess.inputs = [table_a]
# newProcess.outputs = [table_b]
# newProcess.code = "select {{ var(msg) }}"
# newProcess.sql = "select 'hello world'"
# newProcess.column_processes = None
# newProcess.description = 'testing a dbt processes between SPRINT_USER_METRICS model/view'
# response = client.asset.save(newProcess)

# atlan_asset = client.get_asset_by_guid(guid='e7d4ce83-8808-4a62-a224-7e8d2459fd35', asset_type=Table)
# print(atlan_asset.dbt_qualified_name)

# atlan_asset.dbt_qualified_name = 'x'
# client.asset.save(atlan_asset)

# atlan_asset = client.get_asset_by_guid(guid='e7d4ce83-8808-4a62-a224-7e8d2459fd35', asset_type=Table)
# print(atlan_asset.dbt_qualified_name)

# x = DbtModel()
# x.tenant_id = 'tel'
# x.guid = '1a111111-1111-11aa-11aa-11a1111a1a11'
# x.qualified_name = 'default/dbt/1723739048/x'
# x.description = 'testing api creating a dbt model'
# x.dbt_alias = 'dbt_alias_test'




In [ ]:
# parameter to be passed in in the Prefect/Astro script
dbt_command = 'dbt run --select tag:sapo --exclude pub.*'


admin_db = os.environ['SNOWFLAKE_ADMIN']
query = f"""
    select *
    from {admin_db}.DBT_ARTIFACTS.TEL_DBT_RUNS
    where INVOCATION_COMMAND = '{dbt_command}'
"""

conn, messages = TT.getSnowFlakeConn('ADMIN', 'CONTROL', [])
df_models = conn.cursor().execute(query).fetch_pandas_all()
df_models

In [167]:
for idx, row in df_models.iterrows():
    invocation_id = row['COMMAND_INVOCATION_ID']
    node_id = row['NODE_ID']
    db = row['DATABASE'].upper()
    schema = row['SCHEMA'].upper()
    alias = row['ALIAS'].upper()
    tags = row['TAGS']
    tags = set(ast.literal_eval(tags))
    dt = row['COMPILE_STARTED_AT']
    unix_seconds = (dt - pd.Timestamp('1970-01-01')) // pd.Timedelta('1milliseconds')
    status = row['STATUS'].title()
    status = status.replace('Error', 'Fail')

    atlan_qname = f'default/snowflake/1650402931/{db}/{schema}/{alias}'
    if db == 'PUB':
        object_updater = View.updater(
            name=alias,
            qualified_name=atlan_qname,
        )
    else:
        object_updater = Table.updater(
            name=alias,
            qualified_name=atlan_qname,
        )

    object_updater.asset_dbt_job_name = invocation_id # uses invocation id
    object_updater.asset_dbt_job_status = status # Success or Fail
    object_updater.asset_dbt_job_last_run = unix_seconds # Unixtime with millisecond
    object_updater.asset_dbt_tags = tags # make sure this doesnt overwrite

    response = client.asset.save(object_updater)


## Metadata Search

### Add "Issue" announcements to failed DQ assets

In [ ]:
request = (
    FluentSearch()  # 
     .where(CompoundQuery.asset_type(Table))  # 
     .where(CompoundQuery.active_assets())
    #  .where(Table.NAME.startswith("abc"))
    #  .sort(Table.UPDATE_TIME.order(SortOrder.DESCENDING))  # 
).to_request()
results = list(client.asset.search(request))
results

In [ ]:
issue = Announcement(
    announcement_title="Caution",
    announcement_message="At least one of the DQ checks failed.",
    announcement_type=AnnouncementType.ISSUE,
)

request = (
    FluentSearch()
    .where(CompoundQuery.active_assets())
    .where(CustomMetadataField(set_name="Data Quality", attribute_name="DQ_CHECKS_PASSED").eq(False))
    .where_not(Asset.ANNOUNCEMENT_TYPE.eq(AnnouncementType.ISSUE.value))
).to_request()
results = list(client.asset.search(request))

for r in results:
    client.asset.update_announcement(
        asset_type=type(r),
        qualified_name=r.qualified_name,
        name=r.name,
        announcement=issue,
    )
    print(f'New anouncement on {r.qualified_name}')


### Removes "Issue" from passed DQ assets

In [ ]:
request = (
    FluentSearch()
    .where(CompoundQuery.active_assets())
    .where_not(CustomMetadataField(set_name="Data Quality", attribute_name="DQ_CHECKS_PASSED").eq(False))
    .where(Asset.ANNOUNCEMENT_TYPE.eq(AnnouncementType.ISSUE.value))
).to_request()
results = list(client.asset.search(request))

for r in results:
    client.asset.remove_announcement( # 
        asset_type=type(r), # 
        qualified_name=r.qualified_name,
        name=r.name
    )
    print(f'Removed Issue from {r.qualified_name}')


## Upstream column mapping to ODS

In [ ]:
asset = client.asset.get_by_guid(
    guid="a2088e9f-dc29-4dba-96af-d3a489d05c43",
    asset_type=Table 
)

ods_mappings = {}
tbl_columns = asset.columns or []

for c in tbl_columns:
    if c.entity_status == 'ACTIVE':
        col_name = c.display_text
        col_guid = c.guid

        request = FluentLineage(
            starting_guid=col_guid,
            depth=1000000000,
            direction=LineageDirection.UPSTREAM,
            size=1000000000,
        ).request

        lineage_response = client.asset.get_lineage_list(request) 

        ods_cols = []
        for r in lineage_response:
            assert r.status.name is not None
            if r.type_name == 'Column' and r.status.name == 'ACTIVE':
                qualified_name = r.attributes.qualified_name or ''
                db, schema, table, col = qualified_name.split('/')[3:]
                full_name = '.'.join([schema, table, col])
                if db == 'ODS':
                    ods_cols.append(full_name)

        ods_mappings[col_name] = sorted(set(ods_cols))

data = [[report_col, ods_col] for report_col, ods_cols in ods_mappings.items() for ods_col in ods_cols]
df = pd.DataFrame(data, columns=['Report Column', 'ODS Column'])
df.to_csv('ods_mappings.csv', index=False)
df

## Downstream column mapping to COMMON.REPORTS

In [ ]:
schema_guid = '98daefc7-4c44-4a09-b93f-e91ef46548ea'

asset = client.asset.get_by_guid(
    guid=schema_guid,
    asset_type=Schema
)

schema_tables = asset.tables or []

reported_columns = {}
for tbl in schema_tables:
    if tbl.type_name=='Table' and tbl.entity_status=='ACTIVE':
        tbl_guid = tbl.guid
        tbl_asset = client.asset.get_by_guid(
            guid=tbl_guid,
            asset_type=Table 
        )
        tbl_name = tbl_asset.attributes.name or 'TBL_ERROR'
        tbl_columns = tbl_asset.columns or []
        
        for col_asset in tbl_columns:
            col_guid = col_asset.guid
            request = FluentLineage(
                starting_guid=col_guid,
                depth=1000000000,
                direction=LineageDirection.DOWNSTREAM,
                size=1000000000,
                where_assets=[
                    Asset.STATUS.in_lineage.eq(EntityStatus.ACTIVE),
                ],
                includes_in_results=[
                    Asset.TYPE_NAME.in_lineage.eq('Column'),
                    Asset.QUALIFIED_NAME.in_lineage.starts_with('default/snowflake/1650402931/PUB/DE_'),
                ],
            ).request
            lineage_response = list(client.asset.get_lineage_list(request))
            
            if lineage_response:
                col_name = col_asset.display_text or 'COL_ERROR'
                col_name = tbl_name + '.' + col_name
                num_reports = len({'/'.join(l.qualified_name.split('/')[:6]) for l in lineage_response})
                reports = sorted({'.'.join(l.qualified_name.split('/')[4:6]) for l in lineage_response})
                reported_columns[col_name] = reports
        print(f'{dt.now()}:  done with {tbl_name}')

with open('reported_columns.json', 'w') as f:
    json.dump(reported_columns, f, indent=4)

print('complete')

## PUB mapping back to ODS sources

In [49]:
pub_sources = {}

search_request = (
    FluentSearch()
    .where(CompoundQuery.active_assets())
    .where(CompoundQuery.asset_type(View))
    .where(Table.QUALIFIED_NAME.startswith('default/snowflake/1650402931/PUB/DE_')) 
).to_request()
pub_results = list(client.asset.search(search_request))

for res in pub_results:
    starting_guid = res.guid
    view_name = res.qualified_name[29:].replace('/', '.')
    request = FluentLineage(
        starting_guid=starting_guid,
        depth=1000000000,
        direction=LineageDirection.UPSTREAM,
        size=1000000000,
        includes_in_results=[
            Asset.TYPE_NAME.in_lineage.eq('Table'),
            Asset.QUALIFIED_NAME.in_lineage.starts_with('default/snowflake/1650402931/ODS'),
        ],
    ).request
    lineage_reults = list(client.asset.get_lineage_list(request))
    schemas = sorted({t.qualified_name.split('/')[4] for t in lineage_reults})
    pub_sources[view_name] = schemas

import json
with open('pub_sources.json', 'w') as f:
    json.dump(pub_sources, f)

## Mapping metadata between architectures

In [ ]:
df = pd.DataFrame()
folder = 'C:\\Users\\223749\\Downloads\\OneDrive_1_2-7-2025\\'
for file in os.listdir(folder):
    file_path = folder + file
    print(file_path)
    if file_path.endswith('.csv'):
        temp_df = pd.read_csv(file_path)
    elif file_path.endswith('.xlsx'):
        temp_df = pd.read_excel(file_path)
    df = pd.concat([df, temp_df])

df = df[['OLD OBJ', 'NEW OBJ']].drop_duplicates()
df

C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_ENGINEERING_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_ENGINEERING_FINAL_MAPPING_DOCUMENT.xlsx
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_FINANCE_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_FINANCE_FINAL_MAPPING_DOCUMENT.xlsx
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_MANUFACTURING_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_MANUFACTURING_FINAL_MAPPING_DOCUMENT.xlsx
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_PSG_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_PSG_FINAL_MAPPING_DOCUMENT.xlsx
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_QUALITY_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_QUALITY_FINAL_MAPPING_DOCUMENT.xlsx
C:\Users\223749\Downloads\OneDrive_1_2-7-2025\TMEA_SAFETY_FINAL_MAPPING_DOCUMENT.csv
C:\Users\223749\Downloads\OneDrive_1_2

,OLD OBJ,NEW OBJ
0,PUB.DE_MANUFACTURING.TMEA_EAR_DASHBOARD,GOLD.TMEA_ENGINEERING.EAR_DASHBOARD
1,COMMON.DE_MANUFACTURING.TMEA_EAR_DASHBOARD,GOLD.TMEA_ENGINEERING.EAR_DASHBOARD
2,PUB.DE_MANUFACTURING.TMEA_ENGINEERING_CHANGE_R...,GOLD.TMEA_ENGINEERING.ENGINEERING_CHANGE_REVIE...
3,COMMON.DE_MANUFACTURING.TMEA_ENGINEERING_CHANG...,GOLD.TMEA_ENGINEERING.ENGINEERING_CHANGE_REVIE...
4,PUB.DE_MANUFACTURING.TMEA_ENGINEERING_INTERNAL...,GOLD.TMEA_ENGINEERING.ENGINEERING_INTERNAL_ORD...
...,...,...
27,COMMON.DE_MANUFACTURING.TMEA_MATERIAL_DOCUMENT...,GOLD.TMEA_SUPPLY_CHAIN.MATERIAL_DOCUMENT_LIST
36,COMMON.DE_MANUFACTURING.TMEA_PRODUCTION_VALUE_...,GOLD.TMEA_SUPPLY_CHAIN.PRODUCTION_VALUE_STREAM
74,COMMON.DE_MANUFACTURING.TMEA_QUALITY_GOOD_RECE...,GOLD.TMEA_SUPPLY_CHAIN.GOOD_RECEIPT_RECEIVABLES
75,COMMON.DE_MANUFACTURING.TMEA_QUALITY_MRB_STOCK,GOLD.TMEA_SUPPLY_CHAIN.MRB_STOCK


In [ ]:
for new_obj in sorted(set(df['NEW OBJ'])):
    asset_type = View if new_obj.startswith('PUB.') else Table
    new_qname = 'default/snowflake/1650402931/' + new_obj.replace('.', '/')
    try:
        new_asset = client.asset.get_by_qualified_name(qualified_name=new_qname, asset_type=asset_type)
    except:
        print(f'Failed to find {new_qname} in Atlan.')
        continue
    df_filt = df[df['NEW OBJ'] == new_obj]

    old_assets = []
    for old_obj in sorted(set(df_filt['OLD OBJ'])):
        pub_found = False
        old_qname = 'default/snowflake/1650402931/' + old_obj.replace('.', '/')
        old_db = old_obj.split('.')[0]
        if old_db == 'PUB':
            pub_found = True
            asset_type = View
        else:
            asset_type = Table
        old_asset = client.asset.get_by_qualified_name(qualified_name=old_qname, asset_type=asset_type)
        user_desc_found = True if old_asset.user_description else False
        sf_desc_found = True if old_asset.description else False

        asset_info = {}
        asset_info['guid'] = old_asset.guid
        asset_info['qualified_name'] = old_qname
        asset_info['user_desc_found'] = user_desc_found
        asset_info['pub_found'] = pub_found
        asset_info['sf_desc_found'] = sf_desc_found
        asset_info['certificate_verified'] = str(old_asset.certificate_status) == 'CertificateStatus.VERIFIED'
        asset_info['update_time'] = old_asset.update_time
        asset_info['asset_instance'] = old_asset
        old_assets.append(asset_info)

    ordered_assets = sorted(old_assets, key=lambda a: (a['user_desc_found'], a['pub_found'], a['sf_desc_found'], a['certificate_verified'], a['update_time']), reverse=True)
    best_old_asset = ordered_assets[0]['asset_instance']
    print(f'Matched {new_asset.name} to {best_old_asset.name}')
    
    assert new_asset.name and new_asset.qualified_name
    object_updater = Table.updater(
        name=new_asset.name,
        qualified_name=new_asset.qualified_name,
    )
    object_updater.description = best_old_asset.description
    if best_old_asset.user_description:
        object_updater.user_description = best_old_asset.user_description
    response = client.asset.save(object_updater)

Matched SAPO_CATS_DATABASE_TABLE_FOR_TIME_SHEET to CATS_DATABASE_TABLE_FOR_TIME_SHEET
Matched SAPO_HR_MASTER_RECORD_INFOTYPE_0001 to HR_MASTER_RECORD_INFOTYPE_0001
Matched SAPO_ORDER_MASTER_DATA to ORDER_MASTER_DATA
Matched SAPO_PRODUCTION_SUPERVISOR to PRODUCTION_SUPERVISOR
Matched SHAREPOINT_ENGINEERING_ACTION_REQUEST to ENGINEERING_ACTION_REQUEST
Matched SHAREPOINT_ENGINEERING_ACTION_REQUEST_HIST to ENGINEERING_ACTION_REQUEST_HIST
Matched SAPO_ACTUAL_LINE_ITEMS to ACTUAL_LINE_ITEMS
Matched SAPO_BILLING_DOCUMENT_ITEM_DATA to BILLING_DOCUMENT_ITEM_DATA
Matched SAPO_COST_CENTER_MASTER_RECORD to COST_CENTER_MASTER_RECORD
Matched SAPO_COST_CENTER_TEXTS to COST_CENTER_TEXTS
Matched SAPO_CO_OBJECT_COST_TOTALS_FOR_EXTERNAL_POSTINGS to CO_OBJECT_COST_TOTALS_FOR_EXTERNAL_POSTINGS
Matched SAPO_CO_OBJECT_COST_TOTALS_FOR_INTERNAL_POSTINGS to CO_OBJECT_COST_TOTALS_FOR_INTERNAL_POSTINGS
Matched SAPO_CO_OBJECT_DOCUMENT_HEADER to CO_OBJECT_DOCUMENT_HEADER
Matched SAPO_CO_OBJECT_LINE_ITEMS_BY_PERIOD 